# Audit the AI-written pipeline

A colleague asked a chat assistant for "a churn model in scikit-learn" and pasted the answer below. It runs and prints an accuracy of 0.82. You are the reviewer: find the four faults that make that number untrustworthy, and measure what each one hides.

One row is one telco customer; `Churn` is `Yes` when the customer left in the last month. Work in pairs: one person types, the other reads the code and asks "which rows did that line learn from?". Swap after step 3.

In [1]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## Step 1 · Read the code before you run it

For every line in the next cell, write whether it **learns** something from the data (a mean, a list of columns, weights) and, if so, from which rows: all rows, or training rows only. Write your notes in the cell after it.

## Step 2 · Run it

Note the accuracy and the number of test customers.

In [2]:
# The AI-written pipeline, as given
df = pd.read_csv("data/telco_churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
y = (df["Churn"] == "Yes").astype(int)
X = pd.get_dummies(df.drop(columns=["customerID", "Churn"]))
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("Accuracy:", accuracy_score(y_test, model.predict(X_test)))

Accuracy: 0.8204400283889283


In [3]:
# Lines that learn: get_dummies learns the list of categories (all rows); StandardScaler learns
# each column's mean and standard deviation (all rows, before the split); LogisticRegression
# learns weights (training rows). to_numeric and the split learn nothing.
X_raw = pd.get_dummies(df.drop(columns=["customerID", "Churn"]))     # the frame before scaling
print(len(y_test), "test customers,", X_raw.shape[1], "columns after get_dummies")

1409 test customers, 45 columns after get_dummies


## Step 3 · Finding 1: did anything learn from the test customers?

Compare the mean of `tenure` over all rows with its mean over the training rows only. Then rewrite the code so the scaler sits inside a `make_pipeline` that is fitted on the training rows only. Does the accuracy move?

In [4]:
Xr_train, Xr_test = train_test_split(X_raw, test_size=0.2, random_state=42)    # same rows as above
print("tenure mean, all rows:     ", round(X_raw["tenure"].mean(), 2))
print("tenure mean, training rows:", round(Xr_train["tenure"].mean(), 2))
fixed = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xr_train, y_train)
fixed_acc = accuracy_score(y_test, fixed.predict(Xr_test))
print("accuracy, scaler fitted on the training rows:", round(fixed_acc, 3))

tenure mean, all rows:      32.37
tenure mean, training rows: 32.37
accuracy, scaler fitted on the training rows: 0.82


## Step 4 · Finding 2: is 0.82 good?

Score a `DummyClassifier(strategy="most_frequent")` on the same split. Then count how many of the test customers who really churned the model flagged: `((pred == 1) & (y_test == 1)).sum()` against `y_test.sum()`.

In [5]:
pred = model.predict(X_test)
dummy_acc = accuracy_score(y_test, DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test))
found = ((pred == 1) & (y_test == 1)).sum()
print("dummy accuracy:", round(dummy_acc, 3))
print(f"churners flagged: {found} of {y_test.sum()}")

dummy accuracy: 0.735
churners flagged: 223 of 373


## Step 5 · Finding 3: who decided which customers count as "will churn"?

`predict()` flags a customer when the predicted probability is above 0.5. Print how many test customers are flagged at 0.5, then use `predict_proba(X_test)[:, 1] >= 0.3` and count again: how many flagged, how many of them really churned?

In [6]:
proba = model.predict_proba(X_test)[:, 1]
at_05, at_03 = proba >= 0.5, proba >= 0.3
print("flagged at 0.5:", at_05.sum())
print("flagged at 0.3:", at_03.sum(), "| real churners among them:", (at_03 & (y_test == 1)).sum())

flagged at 0.5: 326
flagged at 0.3: 540 | real churners among them: 299


## Step 6 · Finding 4: would another split give the same number?

Run the corrected pipeline from step 3 with `random_state` 0, 1, 2, 3 and 4 and print the five accuracies.

Then score the course pipeline from D01 5.6 with 5-fold cross-validation on its training rows. The next cell builds it for you: `TotalCharges` converted, the `ColumnTransformer` `pre`, and the stratified 80 / 20 split as `Xc_train`, `yc_train`. Print the mean and the standard deviation.

In [7]:
# The course pipeline from D01 5.6, with its stratified 80 / 20 split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder

course = pd.read_csv("data/telco_churn.csv")
course["TotalCharges"] = pd.to_numeric(course["TotalCharges"], errors="coerce")
yc = (course["Churn"] == "Yes").astype(int)
Xc = course.drop(columns=["customerID", "Churn"])
num_cols = list(Xc.select_dtypes(include="number").columns)
cat_cols = list(Xc.select_dtypes(include=["str", "object"]).columns)
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.20, stratify=yc, random_state=42)
pipe = make_pipeline(pre, LogisticRegression(max_iter=1000))

In [8]:
five = []
for rs in range(5):
    a_train, a_test, b_train, b_test = train_test_split(X_raw, y, test_size=0.2, random_state=rs)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(a_train, b_train)
    five.append(accuracy_score(b_test, m.predict(a_test)))
print("five splits:", ", ".join(f"{a:.3f}" for a in five))

cv_acc = cross_val_score(pipe, Xc_train, yc_train, cv=StratifiedKFold(5, shuffle=True, random_state=42))
print(f"course pipeline, 5-fold CV accuracy {cv_acc.mean():.3f} ± {cv_acc.std():.3f}")

five splits: 0.798, 0.812, 0.807, 0.817, 0.791


course pipeline, 5-fold CV accuracy 0.802 ± 0.012


## Step 7 · The review comment

Four lines for your colleague, each naming the code line, what it hides, and the fix.

In [9]:
missed = y_test.sum() - found
print(f"""Sample review comment
1. `X = StandardScaler().fit_transform(X)` runs before the split, so the scaler learned from the test
   customers. It did not move the score here ({fixed_acc:.3f} after the fix), but the same pattern with a
   target encoder or feature selection would. Fix: put the scaler in a pipeline fitted on the training rows.
2. `accuracy_score` is the only number. A model that says "stays" for everyone scores {dummy_acc:.3f}, and
   this one misses {missed} of the {y_test.sum()} churners. Fix: report recall and precision for churn.
3. `model.predict` flags at a probability of 0.5, a threshold nobody chose. Fix: choose it from the cost
   of a missed churner against the cost of an offer (D03 2.2).
4. One `train_test_split` gives one number. Five seeds give {min(five):.3f} to {max(five):.3f}. Fix: report
   the cross-validated mean and standard deviation ({cv_acc.mean():.3f} ± {cv_acc.std():.3f}).""")

Sample review comment
1. `X = StandardScaler().fit_transform(X)` runs before the split, so the scaler learned from the test
   customers. It did not move the score here (0.820 after the fix), but the same pattern with a
   target encoder or feature selection would. Fix: put the scaler in a pipeline fitted on the training rows.
2. `accuracy_score` is the only number. A model that says "stays" for everyone scores 0.735, and
   this one misses 150 of the 373 churners. Fix: report recall and precision for churn.
3. `model.predict` flags at a probability of 0.5, a threshold nobody chose. Fix: choose it from the cost
   of a missed churner against the cost of an offer (D03 2.2).
4. One `train_test_split` gives one number. Five seeds give 0.791 to 0.817. Fix: report
   the cross-validated mean and standard deviation (0.802 ± 0.012).


## Check yourself

In [10]:
check = {
    "Step 2 · accuracy": round(accuracy_score(y_test, model.predict(X_test)), 3),
    "Step 2 · test customers": len(y_test),
    "Step 2 · columns after get_dummies": X_raw.shape[1],
    "Step 3 · tenure mean, all rows": round(X_raw["tenure"].mean(), 2),
    "Step 3 · tenure mean, training rows": round(Xr_train["tenure"].mean(), 2),
    "Step 3 · accuracy after the fix": round(fixed_acc, 3),
    "Step 4 · dummy accuracy": round(dummy_acc, 3),
    "Step 4 · churners flagged": found,
    "Step 4 · churners in the test rows": y_test.sum(),
    "Step 5 · flagged at 0.5": at_05.sum(),
    "Step 5 · flagged at 0.3": at_03.sum(),
    "Step 5 · churners among them": (at_03 & (y_test == 1)).sum(),
    "Step 6 · five accuracies": ", ".join(f"{a:.3f}" for a in five),
    "Step 6 · CV accuracy mean": round(cv_acc.mean(), 3),
    "Step 6 · CV accuracy standard deviation": round(cv_acc.std(), 3),
}
for label, value in check.items():
    print(f"{label:<42} {value}")

Step 2 · accuracy                          0.82
Step 2 · test customers                    1409
Step 2 · columns after get_dummies         45
Step 3 · tenure mean, all rows             32.37
Step 3 · tenure mean, training rows        32.37
Step 3 · accuracy after the fix            0.82
Step 4 · dummy accuracy                    0.735
Step 4 · churners flagged                  223
Step 4 · churners in the test rows         373
Step 5 · flagged at 0.5                    326
Step 5 · flagged at 0.3                    540
Step 5 · churners among them               299
Step 6 · five accuracies                   0.798, 0.812, 0.807, 0.817, 0.791
Step 6 · CV accuracy mean                  0.802
Step 6 · CV accuracy standard deviation    0.012
